# **R/S BENCHMARK — RUL DATASET GENERATION**

## Prerequisite

Run [`01_generate_dataset.ipynb`](01_generate_dataset.ipynb) $\to$
[`02_train_pce.ipynb`](02_train_pce.ipynb) $\to$
[`03_generate_dataset_nn.ipynb`](03_generate_dataset_nn.ipynb) $\to$
[`04_train_nn.ipynb`](04_train_nn.ipynb) first — this notebook loads the global NN
(`lambda 1`/`lambda 2` vs. $(R, S, t)$) that stage 4 writes.

## What this notebook does

Fixes a single design point $(R, S)$ and sweeps a list of time steps through the trained NN, giving
$\lambda_1(t)$ and $\lambda_2(t)$ at each one **directly** — no need to pick a per-time-step PCE
first. $\lambda_3$ and $\lambda_4$ barely move across the design space (see the $\lambda$ maps in
[`01_plot_lambda_maps.ipynb`](01_plot_lambda_maps.ipynb)), which is why they were never modelled by
the NN — they're fixed here instead, to a value you set or, by default, the mean over the NN's own
training dataset.

At each time step, `generate_rul_dataset_benchmark` builds a
`GlamFKML(lam1, lam2, lam3, lam4)` and draws `n_glam_samples` Monte Carlo realisations of the state
limit function $g$ — the raw material for the spaghetti plot and the RUL analysis in
[`06_plot_rul_analysis.ipynb`](06_plot_rul_analysis.ipynb).

## 1. Libraries

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import dill
import numpy as np
import pandas as pd

from functions import *

/home/casa-wand/Documentos/2024-1_victor_hugo_renata_maria/.venv/lib/python3.11/site-packages/UQpy/__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## 2. Config

`n_latent_samples` must match [`04_train_nn.ipynb`](04_train_nn.ipynb) — it names the NN model
files being loaded.

`lambda3_fixed`/`lambda4_fixed` left as `None` fall back to the mean of `lambda 3`/`lambda 4` over
the NN training dataset; set either explicitly to override.

In [2]:
n_latent_samples = 2500   # must match stage 4 — it names the NN model files being loaded

r_fixed = 5.0   # fixed R for the whole sweep
s_fixed = 2.0   # fixed S for the whole sweep
times   = np.arange(0, 151, 10)   # time steps to sweep through the NN — any grid, not just the training one

lambda3_fixed = None   # None = mean of lambda 3 over the NN training dataset
lambda4_fixed = None   # None = mean of lambda 4 over the NN training dataset

n_glam_samples = 10000   # Monte Carlo samples drawn from the GLD at each time step

print(f"Sweeping t = {times.min()}..{times.max()} years at R={r_fixed}, S={s_fixed}")

Sweeping t = 0..150 years at R=5.0, S=2.0


## 3. Predict lambda 1/2, fix lambda 3/4, and draw the GLD samples

In [3]:
print("="*60)
print("GENERATING THE RUL DATASET")
print("="*60)

result = generate_rul_dataset_benchmark(
                                           r=r_fixed,
                                           s=s_fixed,
                                           times=times,
                                           n_latent_samples=n_latent_samples,
                                           lambda3_fixed=lambda3_fixed,
                                           lambda4_fixed=lambda4_fixed,
                                           n_glam_samples=n_glam_samples,
                                           input_dir='.',
                                           output_dir='.',
                                        )

lambda_df = result['lambda_df']
samples   = result['samples']
print(f"\nSamples shape: {samples.shape}  (n_glam_samples x len(times))")
lambda_df

GENERATING THE RUL DATASET

----------------------------------------
GENERATING RUL DATASET AT R=5.0, S=2.0
----------------------------------------
  lambda 3 fixed at 0.1341, lambda 4 fixed at 0.1285
  t = 0.0: lambda 1 = 3.003, lambda 2 = 6.136, sample mean = 3.004, sample std = 0.237
  t = 10.0: lambda 1 = 2.658, lambda 2 = 6.238, sample mean = 2.659, sample std = 0.234
  t = 20.0: lambda 1 = 2.311, lambda 2 = 6.328, sample mean = 2.311, sample std = 0.230
  t = 30.0: lambda 1 = 1.953, lambda 2 = 6.693, sample mean = 1.953, sample std = 0.218
  t = 40.0: lambda 1 = 1.602, lambda 2 = 6.917, sample mean = 1.602, sample std = 0.211
  t = 50.0: lambda 1 = 1.263, lambda 2 = 7.100, sample mean = 1.263, sample std = 0.205
  t = 60.0: lambda 1 = 0.912, lambda 2 = 7.083, sample mean = 0.912, sample std = 0.206
  t = 70.0: lambda 1 = 0.565, lambda 2 = 7.160, sample mean = 0.565, sample std = 0.203
  t = 80.0: lambda 1 = 0.200, lambda 2 = 7.749, sample mean = 0.201, sample std = 0.188
  t = 9

,r,s,Time (years),lambda 1,lambda 2,lambda 3,lambda 4
0,5.0,2.0,0.0,3.003417,6.135591,0.134146,0.128459
1,5.0,2.0,10.0,2.658036,6.238472,0.134146,0.128459
2,5.0,2.0,20.0,2.310753,6.328487,0.134146,0.128459
3,5.0,2.0,30.0,1.952825,6.692838,0.134146,0.128459
4,5.0,2.0,40.0,1.601684,6.916609,0.134146,0.128459
5,5.0,2.0,50.0,1.262719,7.099942,0.134146,0.128459
6,5.0,2.0,60.0,0.911773,7.082962,0.134146,0.128459
7,5.0,2.0,70.0,0.564519,7.160044,0.134146,0.128459
8,5.0,2.0,80.0,0.200263,7.749033,0.134146,0.128459
9,5.0,2.0,90.0,-0.153479,7.798964,0.134146,0.128459


## 4. Sanity check

In [4]:
summary = pd.DataFrame({
                           'Time (years)': result['times'],
                           'Sample mean':  samples.mean(axis=0),
                           'Sample std':   samples.std(axis=0),
                           'P(g <= 0)':    (samples <= 0).mean(axis=0),
                        })
summary

,Time (years),Sample mean,Sample std,P(g <= 0)
0,0.0,3.004128,0.237468,0.0000
1,10.0,2.658735,0.233552,0.0000
2,20.0,2.311443,0.230230,0.0000
3,30.0,1.953477,0.217696,0.0000
4,40.0,1.602315,0.210653,0.0000
5,50.0,1.263334,0.205214,0.0000
6,60.0,0.912389,0.205706,0.0000
7,70.0,0.565128,0.203491,0.0020
8,80.0,0.200826,0.188024,0.1434
9,90.0,-0.152920,0.186820,0.7934
